In [1]:
import torch
print(torch.__version__)
dataset_path = "../data/raw"

2.11.0+cu128


In [2]:
x = torch.tensor([1,2,3,4])
print(x)
print(type(x))

tensor([1, 2, 3, 4])
<class 'torch.Tensor'>


In [3]:
y = torch.tensor([
    [1,2,3],
    [4,5,6]
])
print(y.shape)
print(y[0])
print(y[1])
print(y[0][0])
print(x.shape)

torch.Size([2, 3])
tensor([1, 2, 3])
tensor([4, 5, 6])
tensor(1)
torch.Size([4])


In [4]:
from torchvision.datasets import ImageFolder
dataset = ImageFolder(dataset_path)

In [5]:
print(len(dataset))
print(dataset.class_to_idx)

20638
{'Pepper__bell___Bacterial_spot': 0, 'Pepper__bell___healthy': 1, 'Potato___Early_blight': 2, 'Potato___Late_blight': 3, 'Potato___healthy': 4, 'Tomato_Bacterial_spot': 5, 'Tomato_Early_blight': 6, 'Tomato_Late_blight': 7, 'Tomato_Leaf_Mold': 8, 'Tomato_Septoria_leaf_spot': 9, 'Tomato_Spider_mites_Two_spotted_spider_mite': 10, 'Tomato__Target_Spot': 11, 'Tomato__Tomato_YellowLeaf__Curl_Virus': 12, 'Tomato__Tomato_mosaic_virus': 13, 'Tomato_healthy': 14}


In [6]:
img,label = dataset[0]
print(type(img))
print(label)

<class 'PIL.Image.Image'>
0


In [7]:
print(dataset.classes[label])

Pepper__bell___Bacterial_spot


In [8]:
from torchvision import transforms
transform  = transforms.ToTensor()
dataset = ImageFolder(
    dataset_path,transform= transform
)
img,label = dataset[0]
print(type(img))
print(img.shape)
print(label)

<class 'torch.Tensor'>
torch.Size([3, 256, 256])
0


In [9]:
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
1
NVIDIA GeForce RTX 3050 Laptop GPU


In [10]:
from torch.utils.data import DataLoader
loader = DataLoader(
    dataset,
    batch_size= 32,
    shuffle= True
)
images,labels = next(iter(loader))
print(images.shape)
print(labels.shape)
print(labels[:10])

torch.Size([32, 3, 256, 256])
torch.Size([32])
tensor([ 1, 12, 10, 12, 14, 13,  3,  5, 12,  7])


In [11]:
import torch.nn as nn
layer = nn.Linear(5,3)
x = torch.tensor(
    [[1,2,3,4,5]],
    dtype = torch.float32
)
output = layer(x)

print(x.shape)
print(output.shape)
print(output)

torch.Size([1, 5])
torch.Size([1, 3])
tensor([[-2.4863, -0.1246,  2.2297]], grad_fn=<AddmmBackward0>)


In [12]:
print(nn)

<module 'torch.nn' from 'c:\\project\\crop-disease-detector\\.venv\\Lib\\site-packages\\torch\\nn\\__init__.py'>


In [13]:
conv = nn.Conv2d(
    in_channels = 3,
    out_channels=  32,
    kernel_size= 3
) 
x = torch.randn(32,3,256,256)
output = conv(x);
print(x.shape)
print(output.shape)

torch.Size([32, 3, 256, 256])
torch.Size([32, 32, 254, 254])


In [14]:
relu = nn.ReLU();
y = torch.tensor([-5,-6,0,3,7])
print(relu(y))

tensor([0, 0, 0, 3, 7])


In [15]:
max = nn.MaxPool2d(kernel_size=2)
print(max(x).shape)

torch.Size([32, 3, 128, 128])


In [16]:
flat = nn.Flatten()
print(flat(x).shape)

torch.Size([32, 196608])


In [17]:

class CropDisease(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3,32,3)
        self.bn1 = nn.BatchNorm2d(32)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2)

        self.conv2 = nn.Conv2d(32,64,3)
        self.bn2 = nn.BatchNorm2d(64)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2)

        self.conv3 = nn.Conv2d(64,128,3)
        self.bn3 = nn.BatchNorm2d(128)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(2)

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.flat = nn.Flatten()
        self.fc1 = nn.Linear(128,128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128,15)

    
    def forward(self,x):
        x = self.pool1(self.relu1(self.bn1(self.conv1(x))))
        x = self.pool2(self.relu2(self.bn2(self.conv2(x))))
        x = self.pool3(self.relu3(self.bn3(self.conv3(x))))
        x = self.gap(x)
        x = self.flat(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

In [18]:
model = CropDisease()

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print(total_params)

112143


In [19]:
model = CropDisease()
print(model)

CropDisease(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu1): ReLU()
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu2): ReLU()
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1))
  (bn3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu3): ReLU()
  (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (gap): AdaptiveAvgPool2d(output_size=1)
  (flat): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=128, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=15, bias=True)

In [20]:
print(model(x).shape)

torch.Size([32, 15])


In [21]:
lossFunction = nn.CrossEntropyLoss()
print(lossFunction)

CrossEntropyLoss()


In [22]:
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
print(optimizer)

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [23]:
for images,labels in loader:
    print(images.shape)
    print(labels.shape)
    break

torch.Size([32, 3, 256, 256])
torch.Size([32])


In [24]:
output = model(images)
loss = lossFunction(output,labels)
print(loss)

tensor(2.6892, grad_fn=<NllLossBackward0>)


In [25]:
optimizer.zero_grad
loss.backward()
optimizer.step()
print("done")

done


In [26]:
print(len(dataset))

20638


In [27]:
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size,test_size]
)

In [28]:
print(len(train_dataset))
print(len(test_dataset))

16510
4128


In [29]:
train_loader = DataLoader(
    train_dataset,
    batch_size= 32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size= 32,
    shuffle=False
)

In [30]:
len(test_loader)

129

In [31]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [32]:
model = CropDisease().to(device)

print(next(model.parameters()).device)

cuda:0


In [33]:
lossFunction = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [34]:
lossFunction = nn.CrossEntropyLoss()

In [35]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [36]:
for images, labels in train_loader:
    images = images.to(device)
    labels = labels.to(device)

    outputs = model(images)

    print(outputs.shape)
    print(labels.shape)

    break

torch.Size([32, 15])
torch.Size([32])


In [37]:
train_transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.5,0.5,0.5],
        [0.5,0.5,0.5]
    )
])

test_tranform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.5,0.5,0.5],
        [0.5,0.5,0.5]
    )
])

In [38]:
base_dataset = ImageFolder(dataset_path)
train_size = int(0.8 * len(base_dataset))
test_size = len(base_dataset) - train_size

train_subset, test_subset = random_split(
    base_dataset,
    [train_size,test_size]
)

In [39]:
train_dataset = ImageFolder(
    dataset_path,
    transform=train_transform
)

test_dataset = ImageFolder(
    dataset_path,
    transform=test_tranform
)

In [40]:
from torch.utils.data import Subset

train_dataset = Subset(
    train_dataset,
    train_subset.indices
)

test_dataset = Subset(
    test_dataset,
    test_subset.indices
)

In [41]:
train_loader = DataLoader(
    train_dataset,
    batch_size= 32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size= 32,
    shuffle=False
)

In [42]:
model = CropDisease().to(device)
lossFunction = nn.CrossEntropyLoss()
from torch.optim import Adam
optimizer = Adam(
    model.parameters(),
    lr=0.001
)

In [43]:
epochs = 10
for epoch in range(epochs):
    model.train()

    running_loss = 0
    
    for images,labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = lossFunction(outputs,labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss/len(train_loader)
    print("Epoch", epoch+1,"Average Loss :",avg_loss)

Epoch 1 Average Loss : 1.35906770582809
Epoch 2 Average Loss : 0.810037526684676
Epoch 3 Average Loss : 0.6461170679608057
Epoch 4 Average Loss : 0.5438454791383688
Epoch 5 Average Loss : 0.46433796264346716
Epoch 6 Average Loss : 0.40030071947925777
Epoch 7 Average Loss : 0.3807716810166143
Epoch 8 Average Loss : 0.3436470022641642
Epoch 9 Average Loss : 0.3245879838228688
Epoch 10 Average Loss : 0.30243969888448025


In [44]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images,labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predictions = torch.argmax(outputs,dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
accuracy = (correct / total) * 100
print("Test Accuracy :", accuracy)

Test Accuracy : 93.4108527131783


In [45]:
all_predictions = []
all_labels = []

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        predictions = torch.argmax(outputs, dim=1)

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

In [46]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    all_labels,
    all_predictions
)
for i in range(len(cm)):
    accuracy = cm[i][i]/cm[i].sum()*100
    print("Class",i,"Accuracy :",accuracy,"%")

Class 0 Accuracy : 76.92307692307693 %
Class 1 Accuracy : 99.07120743034056 %
Class 2 Accuracy : 96.17224880382776 %
Class 3 Accuracy : 85.22167487684729 %
Class 4 Accuracy : 89.28571428571429 %
Class 5 Accuracy : 95.02369668246445 %
Class 6 Accuracy : 77.83505154639175 %
Class 7 Accuracy : 87.63440860215054 %
Class 8 Accuracy : 94.6236559139785 %
Class 9 Accuracy : 97.78393351800554 %
Class 10 Accuracy : 95.29411764705881 %
Class 11 Accuracy : 90.74074074074075 %
Class 12 Accuracy : 98.09523809523809 %
Class 13 Accuracy : 100.0 %
Class 14 Accuracy : 99.37888198757764 %


In [47]:
torch.save(
    model.state_dict(),
    "model.pth"
)

In [1]:
from torchvision import models
model = models.resnet18(weights="DEFAULT")
print(model)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\shukl/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100.0%


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [ ]:
import torch.nn as nn
model.fc = nn.Linear(512,15)
print(model.fc)

Linear(in_features=512, out_features=15, bias=True)


In [4]:
total_params = sum(p.numel() for p in model.parameters())
print(total_params)

11184207
